### https://www.knime.com/blog/data-science-glossary

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://www.knime.com/blog/data-science-glossary"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Ambil semua istilah (biasanya dalam <h2> atau <strong>)
keywords = []
for tag in soup.find_all(["h2", "strong"]):
    text = tag.get_text(strip=True)
    if text:
        keywords.append(text)

print(f"Jumlah keyword: {len(keywords)}")
keywords_df = pd.DataFrame(keywords[3:-2], columns=["keyword"])
keywords_df.to_csv("dataset/main_data_llm/knime_skills.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd
import re

# Baca file CSV
df = pd.read_csv("dataset/archived/06_skills.csv")

# Daftar keyword (gabungan IT + Data Science)
keywords = [
    "computer science","informatics","software engineering","programming","algorithms","data structures",
    "operating systems","networking","cybersecurity","cryptography","cloud computing","devops","agile",
    "java","c++","c#","python","php","javascript","html","css","sql",
    "data science","machine learning","deep learning","neural networks","artificial intelligence",
    "regression","classification","clustering","text mining","data mining","big data",
    "pandas","numpy","scikit-learn","tensorflow","pytorch","keras","spark","hadoop",
    "tableau","power bi","grafana","data visualization","etl","data warehouse"
]

# Normalisasi teks
def normalize(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip()

# Filter baris relevan
mask = df.apply(lambda row: any(kw in normalize(" ".join(map(str,row))) for kw in keywords), axis=1)
df_filtered = df[mask]

def word_count(text):
    return len(str(text).split())

df_top_clean = df_filtered[
    (df_filtered['skill'].apply(word_count) <= 3) &
    (~df_filtered['skill'].str.contains(r"[\"']", regex=True))
]

df_top_clean.to_csv("skills_top_clean.csv", index=False)

In [3]:
import pandas as pd

dataset_job = pd.read_csv("dataset/main_data_llm/job_portal/dataset_jobstreet-job-scraper_2025-12-06_15-43-45-583.csv", sep=",")
# dataset_job.columns
list_job = dataset_job[['company/description', 'jobDescriptionHtml', 'company/name','location','salary','sourceUrl','title','workTypes/0']]
list_job

,company/description,jobDescriptionHtml,company/name,location,salary,sourceUrl,title,workTypes/0
0,"PT. Metrodata Electronics, Tbk","<div class=""_1yd5ljl0 qnrjqb0""><p>We are looki...",Metrodata,Jakarta Raya,NaN,https://id.jobstreet.com/id/Data-Science-jobs,Data Scientist,Full time
1,Kredivo Group,"<div class=""_1yd5ljl0 qnrjqb0""><p>As a Data Sc...",Kredivo Group,Jakarta Raya,NaN,https://id.jobstreet.com/id/Data-Science-jobs,Data Scientist,Full time
2,PT Sharing Vision Indonesia,"<div class=""_1yd5ljl0 qnrjqb0""><p><strong>DATA...",Sharing Vision Indonesia,"Jakarta Pusat, Jakarta Raya",NaN,https://id.jobstreet.com/id/Data-Science-jobs,Data Scientist,Kontrak
3,PT. CIPTA ARTHA NADYA,"<div class=""_1yd5ljl0 qnrjqb0""><h2>Key Respons...",New Connected Indonesia,Jakarta Raya,Rp 9.000.000 – Rp 13.000.000 per month,https://id.jobstreet.com/id/Data-Science-jobs,Data Scientist,Full time
4,Infomedia,"<div class=""_1yd5ljl0 qnrjqb0""><strong><b>Job ...",Infomedia,"Kebayoran Lama, Jakarta Raya",NaN,https://id.jobstreet.com/id/Data-Science-jobs,Officer Data Scientist,Kontrak
...,...,...,...,...,...,...,...,...
59,PT Nusantara Compnet Integrator,"<div class=""_1yd5ljl0 qnrjqb0""><p>&nbsp;<stron...",Nusantara Compnet Integrator,Jakarta Raya,NaN,https://id.jobstreet.com/id/informatics-jobs,IT Account Manager Cloud (AWS),Full time
60,PT Indonesia Epson Industry,"<div class=""_1yd5ljl0 qnrjqb0""><p><strong>Resp...",Epson,Jawa Barat,Rp 7.000.000 – Rp 8.000.000 per month,https://id.jobstreet.com/id/informatics-jobs,Firmware Design Staff,Full time
61,PT Lawencon Internasional,"<div class=""_1yd5ljl0 qnrjqb0""><p><strong>Job ...",Lawencon International,"Jakarta Selatan, Jakarta Raya",NaN,https://id.jobstreet.com/id/informatics-jobs,SAP Helpdesk (L1),Full time
62,PT Astra Graphia Information Technology (AGIT),"<div class=""_1yd5ljl0 qnrjqb0""><p><strong>Mini...",Astra Graphia,Jakarta Raya,NaN,https://id.jobstreet.com/id/informatics-jobs,.NET Developer,Kontrak


In [4]:
import re
import html
import pandas as pd
from bs4 import BeautifulSoup
import unicodedata

# ------------- helper cleaning functions -------------
def strip_html(html_text: str) -> str:
    """Remove HTML tags and script/style content, return plain text."""
    if not isinstance(html_text, str):
        return ""
    soup = BeautifulSoup(html_text, "html.parser")
    # remove script/style
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator=" ")
    return text

def remove_urls(text: str) -> str:
    return re.sub(r'https?://\S+|www\.\S+', ' ', text)

def remove_emails(text: str) -> str:
    return re.sub(r'\S+@\S+\.\S+', ' ', text)

def remove_phone_numbers(text: str) -> str:
    # common phone patterns (simple)
    return re.sub(r'(\+?\d{1,3}[\s\-\.])?(\(?\d{2,4}\)?[\s\-\.])?\d{3,4}[\s\-\.]?\d{3,4}', ' ', text)

def remove_html_entities(text: str) -> str:
    return html.unescape(text)

def normalize_unicode(text: str) -> str:
    # normalize accents and similar
    return unicodedata.normalize("NFKC", text)

def remove_extra_whitespace(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()

def remove_non_printable(text: str) -> str:
    return ''.join(ch for ch in text if unicodedata.category(ch)[0] != "C")

def remove_special_chars(text: str, keep_punct: bool = False) -> str:
    if keep_punct:
        # keep basic punctuation .,;:!?()[]-/
        return re.sub(r'[^0-9A-Za-z\u00C0-\u017F\.\,\;\:\!\?\(\)\[\]\-\/\s]', ' ', text)
    else:
        return re.sub(r'[^0-9A-Za-z\u00C0-\u017F\s]', ' ', text)

def clean_text(html_text: str,
               lower: bool = True,
               keep_punct: bool = False) -> str:
    """
    Full cleaning pipeline:
    1. strip HTML and scripts
    2. decode HTML entities
    3. remove URLs, emails, phones
    4. normalize unicode
    5. remove non-printable and special chars
    6. normalize whitespace and optional lowercase
    """
    if not isinstance(html_text, str) or not html_text.strip():
        return ""

    text = strip_html(html_text)
    text = remove_html_entities(text)
    text = remove_urls(text)
    text = remove_emails(text)
    text = remove_phone_numbers(text)
    text = normalize_unicode(text)
    text = remove_non_printable(text)
    text = remove_special_chars(text, keep_punct=keep_punct)
    text = remove_extra_whitespace(text)
    if lower:
        text = text.lower()
    return text

# ------------- apply to dataframe -------------
# load dataset (your path)
dataset_job = pd.read_csv("dataset/main_data_llm/job_portal/dataset_jobstreet-job-scraper_2025-12-06_15-43-45-583.csv", sep=",")

# select columns as before
list_job = dataset_job[['company/description', 'jobDescriptionHtml', 'company/name','location','salary','sourceUrl','title','workTypes/0']].copy()

# create cleaned column
list_job['job_clean'] = list_job['jobDescriptionHtml'].apply(lambda x: clean_text(x, lower=True, keep_punct=False))

# quick preview
print(list_job[['jobDescriptionHtml','job_clean']].head(5).to_string(index=False))

# optional: save cleaned column to csv for later use
list_job[['company/name','title',"company/name",'location','job_clean']].to_csv("dataset/cleaned_job_descriptions.csv", index=False)


In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import json
import numpy as np

# === 1. Load model dan tokenizer dari folder final_model ===
print("⏳ Loading Model...")
model_path = "landing_page/backend/final_bert_model(2)" # Pastikan path ini benar
try:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    # Aggregation strategy 'simple' otomatis menggabungkan B-Label dan I-Label
    ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")
    print("✅ Model Loaded.")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    exit()

# === 2. Load data CSV ===
print("⏳ Loading CSV...")
csv_path = "dataset/main_data_llm/job_portal/cleaned_job_descriptions.csv" # Pastikan path benar
try:
    df = pd.read_csv(csv_path)
    print(f"✅ CSV Loaded: {len(df)} baris.")
except Exception as e:
    print(f"❌ Error loading CSV: {e}")
    exit()

# === 3. Fungsi untuk gabung subword (Cleaning tambahan) ===
def merge_subwords(ner_results):
    merged = []
    current = ""
    current_label = None

    for ent in ner_results:
        word = ent["word"]
        label = ent["entity_group"]

        # Hapus token ## dari BERT tokenizer
        if word.startswith("##"):
            current += word[2:]
        else:
            if current:
                merged.append({"word": current, "entity_group": current_label})
            current = word
            current_label = label

    if current:
        merged.append({"word": current, "entity_group": current_label})

    return merged

# === 4. Normalisasi frasa (Opsional) ===
def normalize_phrases(tokens):
    phrases = []
    skip_next = False
    for i in range(len(tokens)):
        if skip_next:
            skip_next = False
            continue
        # Contoh logic: gabung "big" + "data"
        if i < len(tokens)-1 and tokens[i].lower() == "big" and tokens[i+1].lower() == "data":
            phrases.append("Big Data")
            skip_next = True
        else:
            phrases.append(tokens[i])
    return phrases

# === 5. Ekstraksi entitas ===
def extract_entities(text):
    if not isinstance(text, str) or not text.strip():
        return {"education_required": [], "skills_required": []}
        
    ner_results = ner(text)
    merged = merge_subwords(ner_results)

    skills = []
    education = []

    for ent in merged:
        word = ent["word"].strip()
        label = ent["entity_group"]
        
        # Filter berdasarkan Label NER Anda
        if label == "SKILL":
            skills.append(word)
        elif label in ["EDUCATION", "DEGREE", "MAJOR"]:
            education.append(word)

    # Bersihkan duplikat & normalisasi
    skills = normalize_phrases(list(set(skills)))
    education = normalize_phrases(list(set(education)))

    return {
        "education_required": education,
        "skills_required": skills
    }

# === 6. Proses semua baris CSV (DIPERBAIKI) ===
print("⏳ Processing Jobs (ini mungkin memakan waktu)...")
job_descriptions = []

for index, row in df.iterrows():
    # 1. Ambil Title
    title = str(row.get("title", "Unknown Position")).strip()
    
    # 2. Ambil Company (Cek kolom 'company', jika kosong cek 'name')
    raw_company = row.get("company/name")
    
    # Bersihkan nama company
    company = str(raw_company).strip() if pd.notna(raw_company) else "Unknown Company"

    # 3. Ambil Deskripsi Pekerjaan
    job_text = str(row.get("job_clean", "")).strip()
    
    # 4. Ekstrak Skill & Edu pakai AI
    extracted = extract_entities(job_text)
    
    # 5. Susun Dictionary
    job_description = {
        "id": index, # Opsional: tambah ID biar gampang dilacak
        "title": title,
        "company": company,  # <--- SUDAH DIPERBAIKI
        "education_required": extracted["education_required"],
        "skills_required": extracted["skills_required"]
    }
    job_descriptions.append(job_description)
    
    # Print progress setiap 10 data
    if index % 10 == 0:
        print(f"   Processed {index+1}/{len(df)} jobs...", end="\r")

print(f"\n✅ Selesai memproses {len(job_descriptions)} pekerjaan.")

# === 7. Simpan hasil ke file JSON ===
output_path = "landing_page/backend/extracted_job_descriptions.json"
try:
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(job_descriptions, f, indent=2, ensure_ascii=False)
    print(f"💾 Data berhasil disimpan ke: {output_path}")
except Exception as e:
    print(f"❌ Gagal menyimpan JSON: {e}")

# === 8. Cetak contoh hasil pertama ===
if job_descriptions:
    print("\n--- Contoh Hasil Pertama ---")
    print(json.dumps(job_descriptions[0], indent=2))

d:\ProjekAkhir\Aplikasi\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading Model...


Device set to use cpu


✅ Model Loaded.
⏳ Loading CSV...
✅ CSV Loaded: 64 baris.
⏳ Processing Jobs (ini mungkin memakan waktu)...
   Processed 61/64 jobs...
✅ Selesai memproses 64 pekerjaan.
💾 Data berhasil disimpan ke: landing_page/backend/extracted_job_descriptions.json

--- Contoh Hasil Pertama ---
{
  "id": 0,
  "title": "Data Scientist",
  "company": "Metrodata",
  "education_required": [
    "science"
  ],
  "skills_required": [
    "apply for the",
    "improving existing ones",
    "mathematical statistical and programming skills",
    "demand forecasting product recommendation or basket analysis and other solution improve competencies in",
    "brighter career future",
    "develop",
    "data modelling predictive modelling and",
    "r python or others follow project timeline",
    "improve user experience data scientist",
    "machine learning areat metrodata electronics tbk digital solution provider technology innovator",
    "machine learning area",
    "strong mix of",
    "just offer for the jo

In [19]:
import pandas as pd

dataset_job = pd.read_csv("dataset/main_data_llm/job_portal/dataset_jobstreet-job-scraper_2025-12-06_15-43-45-583.csv", sep=",")
dataset_job.columns

dataset_job["jobDescriptionHtml"][9]
# dataset_job["jobId"][9]

'<div class="_1yd5ljl0 qnrjqb0"><p><strong>Qualifications</strong>:</p><ol><li><p><strong>Bachelor’s or Master’s degree</strong> in Computer Science, Data Science, Statistics, Mathematics, or other quantitative fields.</p></li><li><p><strong>Minimum 5 years of applied Data Science or AI experience</strong> with a proven track record of delivering measurable business impact.</p></li><li><p>Experience in <strong>F&amp;B, retail, or fast-paced startup environments</strong> (plus).</p></li><li><p><strong>Strong Python skills</strong>, including experience with libraries such as <strong>scikit-learn, TensorFlow, PyTorch, XGBoost</strong>, and frameworks like <strong>LangChain or OpenAI API</strong> (plus).</p></li><li><p>Proven experience in <strong>deploying machine-learning models</strong> using ML platforms (e.g., <strong>MLFlow, Vertex AI, Azure ML</strong>, or similar).</p></li><li><p>Strong understanding of <strong>statistics, experiment design, A/B testing, and regression methods</st

In [9]:
import pandas as pd

# Baca file CSV
perguruan_tinggi = pd.read_csv("dataset/main_data_llm/education/LIST UNIVERSITAS.csv")

# Bersihkan kolom 'nama' dari tanda kutip dan koma
perguruan_tinggi['nama'] = perguruan_tinggi['nama'].astype(str).str.replace('"', '', regex=False)
perguruan_tinggi['nama'] = perguruan_tinggi['nama'].str.replace(',', '', regex=False)
perguruan_tinggi['nama'] = perguruan_tinggi['nama'].str.strip()

# Simpan hasil ke file baru
perguruan_tinggi['nama'].to_csv("dataset/main_data_llm/education/list_perguruan_tinggi.csv", index=False, header=True)


In [12]:
program_studi = pd.read_csv("dataset/main_data_llm/education/LIST PROGRAM STUDI.csv")
program_studi['Nama Program Studi (Inggris)'].to_csv("dataset/main_data_llm/education/list_program_studi.csv", index=False, header=True)